# Week 05 — Logistic Regression & Regularization
### Dataset: Heart Disease (`heart.csv`)

This notebook applies **L1**, **L2**, and **ElasticNet** regularized Logistic Regression
to predict presence of heart disease (`target`: 1 = disease, 0 = no disease) from
13 clinical features (age, sex, chest pain type, blood pressure, cholesterol, etc.).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("heart.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset Shape: (1025, 14)

Columns:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [2]:
# ============================================================
# 2. BASIC EXPLORATION
# ============================================================

print(df.info())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df["target"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB
None

Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      

In [3]:
# ============================================================
# 3. TARGET VARIABLE
# ============================================================

# target is already:
# 0 = No Heart Disease
# 1 = Heart Disease

X = df.drop(columns=["target"])
y = df["target"]

# ============================================================
# 4. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (820, 13)
Test shape: (205, 13)


## L1 Regularization

Uses the `liblinear` solver with `penalty='l1'`, which drives less-important feature coefficients to exactly zero (built-in feature selection).

In [4]:
# ============================================================
# L1-REGULARIZED LOGISTIC REGRESSION
# ============================================================

model_l1 = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty='l1', solver='liblinear', C=1.0, max_iter=1000, random_state=42))
])

model_l1.fit(X_train, y_train)

y_pred_l1 = model_l1.predict(X_test)
y_proba_l1 = model_l1.predict_proba(X_test)[:, 1]

print("Accuracy:")
print(accuracy_score(y_test, y_pred_l1))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l1))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_l1, target_names=["No Disease", "Disease"]))

print("\nROC-AUC:")
print(roc_auc_score(y_test, y_proba_l1))

Accuracy:
0.8097560975609757

Confusion Matrix:
[[70 30]
 [ 9 96]]

Classification Report:
              precision    recall  f1-score   support

  No Disease       0.89      0.70      0.78       100
     Disease       0.76      0.91      0.83       105

    accuracy                           0.81       205
   macro avg       0.82      0.81      0.81       205
weighted avg       0.82      0.81      0.81       205


ROC-AUC:
0.9300952380952382


## L2 Regularization

Uses the `lbfgs` solver with `penalty='l2'` (Ridge-style), which shrinks coefficients but rarely sets them exactly to zero.

In [5]:
# ============================================================
# L2-REGULARIZED LOGISTIC REGRESSION
# ============================================================

model_l2 = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty='l2', solver='lbfgs', C=1.0, max_iter=1000, random_state=42))
])

model_l2.fit(X_train, y_train)

y_pred_l2 = model_l2.predict(X_test)
y_proba_l2 = model_l2.predict_proba(X_test)[:, 1]

print("Accuracy:")
print(accuracy_score(y_test, y_pred_l2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l2))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_l2, target_names=["No Disease", "Disease"]))

print("\nROC-AUC:")
print(roc_auc_score(y_test, y_proba_l2))

Accuracy:
0.8097560975609757

Confusion Matrix:
[[70 30]
 [ 9 96]]

Classification Report:
              precision    recall  f1-score   support

  No Disease       0.89      0.70      0.78       100
     Disease       0.76      0.91      0.83       105

    accuracy                           0.81       205
   macro avg       0.82      0.81      0.81       205
weighted avg       0.82      0.81      0.81       205


ROC-AUC:
0.9298095238095239


## ElasticNet Regularization

Combines L1 and L2 penalties (`solver='saga'`, `l1_ratio=0.5`), balancing feature selection with coefficient shrinkage.

In [6]:
# ============================================================
# ELASTICNET-REGULARIZED LOGISTIC REGRESSION
# ============================================================

model_en = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=1.0, max_iter=5000, random_state=42))
])

model_en.fit(X_train, y_train)

y_pred_en = model_en.predict(X_test)
y_proba_en = model_en.predict_proba(X_test)[:, 1]

print("Accuracy:")
print(accuracy_score(y_test, y_pred_en))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_en))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_en, target_names=["No Disease", "Disease"]))

print("\nROC-AUC:")
print(roc_auc_score(y_test, y_proba_en))

Accuracy:
0.8097560975609757

Confusion Matrix:
[[70 30]
 [ 9 96]]

Classification Report:
              precision    recall  f1-score   support

  No Disease       0.89      0.70      0.78       100
     Disease       0.76      0.91      0.83       105

    accuracy                           0.81       205
   macro avg       0.82      0.81      0.81       205
weighted avg       0.82      0.81      0.81       205


ROC-AUC:
0.9301904761904762


## Comparing the Three Models

A quick side-by-side comparison of accuracy and ROC-AUC across L1, L2, and ElasticNet.

In [7]:
results = pd.DataFrame({
    "Model": ["L1 (Lasso)", "L2 (Ridge)", "ElasticNet"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_l1),
        accuracy_score(y_test, y_pred_l2),
        accuracy_score(y_test, y_pred_en),
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_l1),
        roc_auc_score(y_test, y_proba_l2),
        roc_auc_score(y_test, y_proba_en),
    ],
})
results

,Model,Accuracy,ROC-AUC
0,L1 (Lasso),0.809756,0.930095
1,L2 (Ridge),0.809756,0.929810
2,ElasticNet,0.809756,0.930190


## Prediction on New Datapoint

In [8]:
# ============================================================
# PREDICT HEART DISEASE FOR A NEW PATIENT
# ============================================================

new_patient = pd.DataFrame([{
    "age": 58,
    "sex": 1,          # 1 = male, 0 = female
    "cp": 2,           # chest pain type (0-3)
    "trestbps": 130,   # resting blood pressure
    "chol": 246,       # serum cholesterol
    "fbs": 0,          # fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
    "restecg": 1,      # resting ECG results
    "thalach": 152,    # max heart rate achieved
    "exang": 0,        # exercise induced angina (1 = yes, 0 = no)
    "oldpeak": 1.0,    # ST depression induced by exercise
    "slope": 2,        # slope of peak exercise ST segment
    "ca": 0,           # number of major vessels colored by fluoroscopy
    "thal": 2          # thalassemia (1=normal, 2=fixed defect, 3=reversible defect)
}])

new_patient

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,58,1,2,130,246,0,1,152,0,1.0,2,0,2


In [9]:
# ============================================================
# PREDICTION (using the ElasticNet model as the final chosen model)
# ============================================================

pred_class = model_en.predict(new_patient)[0]
pred_proba = model_en.predict_proba(new_patient)[0, 1]

label = "Heart Disease" if pred_class == 1 else "No Heart Disease"

print("Predicted class:", pred_class, f"({label})")
print(f"Predicted probability of heart disease: {pred_proba:.4f}")

Predicted class: 1 (Heart Disease)
Predicted probability of heart disease: 0.8898
